# Why doubling after a loss does not work

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/doubling-after-a-loss)

This notebook reproduces every figure and every number in the article.

## How to run it

| | |
| --- | --- |
| **Run it, nothing to install** | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/doubling-down.ipynb) |
| **Open it in VS Code** | [Open in VS Code](https://vscode.dev/github/Arthurnoo/out-of-sample/blob/main/notebooks/doubling-down.ipynb) |

Colab runs it in your browser. VS Code opens the real editor, also in your
browser, with nothing to install. Either way the next cell installs whatever is
missing, so you can simply **Run All**. The first run downloads prices and takes
a minute or two.

On your own machine instead: `git clone https://github.com/Arthurnoo/out-of-sample`

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option('display.width', 130)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

| Knob | What it does | Try |
| --- | --- | --- |
| `SYMBOL` | the market the bets are built from | `QQQ`, `GLD`, `BTC-USD` |
| `BARRIER` | how far price must move to settle a bet | `0.01`, `0.03` |
| `MAX_DAYS` | give up on a bet after this many days | `10`, `40` |
| `BASE_RISK` | fraction of capital risked on the first bet | `0.005`, `0.02` |
| `WINDOW_BETS` | bets per window in the backtest test | `75`, `300` |

`BASE_RISK` is the one that decides everything. The boundary between surviving
and not is narrow, and on this series it sits between 0.005 and 0.01.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — this is the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────

SYMBOL      = 'SPY'
START       = '2000-01-01'

BARRIER     = 0.015      # a bet settles at +1.5% or -1.5%
MAX_DAYS    = 20         # unless neither is reached in 20 days
BASE_RISK   = 0.01       # fraction of capital risked on the first bet

WINDOW_BETS = 150        # bets per window, roughly two years
N_WINDOWS   = 2000

## 1. What the rule costs to run

To place the bet after `n` losses you must already have lost `2**n - 1` units
and still be able to stake `2**n`. Your capital therefore has to cover
`2**(n+1) - 1` in total.

Invert that and you get the only number that matters: a bankroll of `B` units
buys `log2(B + 1)` bets, and no more.

In [ ]:
rows = [{'losses': n, 'next_bet': 2**n, 'already_lost': 2**n - 1,
         'capital_needed': 2**(n+1) - 1} for n in (1, 5, 10, 15, 20)]
print(pd.DataFrame(rows).to_string(index=False))

def bets_affordable(bankroll_units):
    """Combien de mises un capital permet-il de poser avant d'etre a sec ?"""
    return int(np.floor(np.log2(bankroll_units + 1))) if bankroll_units >= 1 else 0

print()
for B in (1_000, 10_000, 100_000, 1_000_000):
    print(f"{B:>10,} units -> {bets_affordable(B):2d} bets")

## 2. The fatal run arrives, it is only a question of when

With a bankroll that survives `n` losses, a single sequence kills you with
probability `p**n`, and you play many sequences. Over enough of them the
probability of having met one goes to 1, for every starting capital.

In [ ]:
def ruin_prob(bankroll_units, p_loss, sequences):
    """Probabilite d'avoir rencontre AU MOINS UNE FOIS la sequence fatale."""
    n = bets_affordable(bankroll_units)
    return 1.0 if n <= 0 else 1 - (1 - p_loss**n)**sequences

games = np.unique(np.logspace(0, 6, 120).astype(int))
fig, ax = plt.subplots(figsize=(8, 4))
for B, c in zip((1_000, 10_000, 100_000, 1_000_000),
                ['#bcd3e6', '#8fb5d4', '#2d6fa8', '#1b4a72']):
    ax.plot(games, [ruin_prob(B, 0.5, int(g))*100 for g in games], lw=1.9,
            color=c, label=f'{B:,} units')
ax.set_xscale('log'); ax.set_xlabel('sequences played (log scale)')
ax.set_ylabel('chance of hitting the fatal run (%)')
ax.legend(title='starting capital'); plt.show()

for g in (1_000, 100_000):
    print(f"100,000 units, {g:>7,} sequences -> {ruin_prob(100_000, 0.5, g)*100:5.1f}%")

## 3. Turning a market into bets

The rule assumes a bet with two outcomes, so applying it to daily returns is
meaningless: doubling your exposure before a day that moves 1% does not double
anything that matters.

The honest translation is the one people actually trade. Enter at the close,
exit at the first barrier touched, and give up after `MAX_DAYS`. Each bet then
settles at roughly plus or minus one unit of risk.

In [ ]:
def get_prices(symbol, start=START):
    h = yf.Ticker(symbol).history(start=start, auto_adjust=True)
    if h.index.tz is not None:
        h.index = h.index.tz_localize(None)
    return h['Close']


def make_bets(px, barrier=BARRIER, max_days=MAX_DAYS):
    """Decoupe une serie de prix en paris a deux issues."""
    v, idx = px.to_numpy(), px.index
    out, t = [], 0
    while t < len(v) - 1:
        entry = v[t]
        hi, lo = entry*(1+barrier), entry*(1-barrier)
        end = min(t + max_days, len(v) - 1)
        for u in range(t+1, end+1):                 # premier seuil touche
            if v[u] >= hi or v[u] <= lo:
                break
        ret = (v[u] - entry) / entry
        out.append({'entry': idx[t], 'days': u-t,
                    'result': ret/barrier,          # en multiples du risque
                    'won': ret > 0})
        t = u
    return pd.DataFrame(out)


px = get_prices(SYMBOL)
bets = make_bets(px)
res = bets.result.to_numpy()
print(f"{len(px)} days -> {len(bets)} bets")
print(f"win rate      {bets.won.mean()*100:.1f}%")
print(f"mean result   {res.mean():+.4f} units of risk")
print(f"longest losing run  {max((len(list(g)) for k, g in __import__('itertools').groupby(res < 0) if k), default=0)} bets")

## 4. The same bets, three sizes

`run_bets` doubles the risk after a loss and resets it after a win. The
account is marked as ruined the first time it reaches zero, and nothing
compounds after that.

In [ ]:
def run_bets(results, base_risk=BASE_RISK):
    """La martingale sur des paris exprimes en multiples du risque engage."""
    cap, risk, losses = 1.0, base_risk, 0
    eq, ruined_at = np.empty(len(results)), None
    for t, r in enumerate(results):
        if cap <= 0:                     # deja ruine, plus rien ne bouge
            eq[t] = 0.0
            continue
        cap += risk * r                  # +risk si gagne, -risk si perdu
        eq[t] = max(cap, 0.0)
        if cap <= 0 and ruined_at is None:
            ruined_at = t
        losses = losses + 1 if r < 0 else 0
        risk = base_risk * (2**losses)   # on double, ou on revient a la base
    peak = np.maximum.accumulate(np.maximum(eq, 1e-12))
    return {'equity': eq, 'final': eq[-1], 'ruined_at': ruined_at,
            'max_dd_pct': (np.min(eq/peak) - 1)*100}


fig, ax = plt.subplots(figsize=(8, 4.2))
for base, c in zip((0.005, 0.01, 0.02), ['#8fb5d4', '#2d6fa8', '#1b4a72']):
    o = run_bets(res, base)
    ax.plot(np.where(o['equity'] > 0, o['equity'], np.nan), lw=1.7, color=c,
            label=f"doubling, {base*100:g}% base")
    print(f"doubling {base*100:>4.1f}%  final {o['final']:8.3f}  "
          f"worst fall {o['max_dd_pct']:6.1f}%  "
          + (f"RUINED at bet {o['ruined_at']}" if o['ruined_at'] is not None else 'survives'))

flat = np.cumprod(1 + BASE_RISK * res)
ax.plot(flat, lw=1.7, color='#0e8f5f', label=f'flat, {BASE_RISK*100:g}% every bet')
print(f"flat     {BASE_RISK*100:>4.1f}%  final {flat[-1]:8.3f}")
ax.set_yscale('log'); ax.set_xlabel('bet number')
ax.set_ylabel('capital, starting at 1 (log scale)'); ax.legend(); plt.show()

## 5. Why a short backtest looks good

A short window has a good chance of not containing the fatal run. So the
median is flattering and only the left tail tells the truth. Look at where the
outcomes land: there is almost nothing in between.

In [ ]:
rng = np.random.default_rng(0)
starts = rng.integers(0, len(res) - WINDOW_BETS, size=N_WINDOWS)
finals = np.array([run_bets(res[s:s+WINDOW_BETS])['final'] for s in starts])

print(f"{N_WINDOWS} windows of {WINDOW_BETS} bets at {BASE_RISK*100:g}% base risk")
print(f"  median final capital  {np.median(finals):.2f}")
print(f"  ending in profit      {np.mean(finals > 1)*100:.0f}%")
print(f"  ending at zero        {np.mean(finals <= 0)*100:.0f}%")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(finals, 0, np.percentile(finals, 99.5)), bins=60,
        color='#8b98a6', edgecolor='white', lw=0.4)
ax.axvline(1.0, color='#1a2430', ls='--')
ax.set_xlabel('capital at the end of the window'); ax.set_ylabel('windows')
plt.show()

## 6. What the capital actually earns

A won sequence always pays exactly one unit, whatever your bankroll, while the
bankroll you must hold is `2**k - 1`. So protecting yourself better makes the
return on the money tied up worse, not better.

There is an exact result underneath: the number of sequences needed to double
your capital equals the number of sequences before the fatal run, at every
bankroll. The capital is `2**k - 1` and the expected wait for `k` losses in a
row is `2**k`.

In [ ]:
print(pd.DataFrame([
    {'capital_units': 2**k - 1, 'bets_affordable': k, 'gain_per_sequence': 1,
     'return_pct': 100/(2**k - 1), 'one_sequence_in': 2**k}
    for k in (6, 10, 17, 21)
]).to_string(index=False, float_format=lambda v: f'{v:17.5f}'))

print("\nsequences to double vs sequences before ruin")
for k in (10, 17):
    B = 2**k - 1
    print(f"  capital {B:>9,} -> double in {B:>9,}, ruin in {2**k:>9,}, "
          f"ratio {2**k/B:.2f}")

## 7. The comparison that closes it

Two rules cannot be compared at the same base size, because they do not take
the same risk. So we vary the size of each and plot what each one bought: the
capital it ended with, against the worst fall it went through on the way.

Further left is a smaller drawdown and higher is more money, so the better rule
is the one whose curve sits above and to the left of the other.

In [ ]:
dbl = pd.DataFrame([{'risk_pct': b*100, **{k: v for k, v in run_bets(res, b).items()
                                            if k in ('final', 'max_dd_pct')}}
                    for b in (0.0025, 0.005, 0.0075, 0.01)])
flat_rows = []
for b in (0.005, 0.01, 0.02, 0.03, 0.05):
    eq = np.cumprod(1 + b*res)
    flat_rows.append({'risk_pct': b*100, 'final': eq[-1],
                      'max_dd_pct': (eq/np.maximum.accumulate(eq)).min()*100 - 100})
flat_df = pd.DataFrame(flat_rows)
print('doubling:'); print(dbl.to_string(index=False, float_format=lambda v: f'{v:11.2f}'))
print('flat:');     print(flat_df.to_string(index=False, float_format=lambda v: f'{v:11.2f}'))

print(f"\nflat at 2% ends at {flat_df.set_index('risk_pct').loc[2.0,'final']:.1f} "
      f"with a {flat_df.set_index('risk_pct').loc[2.0,'max_dd_pct']:.0f}% fall,")
print(f"doubling at 0.5% ends at {dbl.set_index('risk_pct').loc[0.5,'final']:.1f} "
      f"with a {dbl.set_index('risk_pct').loc[0.5,'max_dd_pct']:.0f}% fall.")

fig, ax = plt.subplots(figsize=(8, 4))
for df, c, lab, mk in ((dbl, '#c83c50', 'doubling after a loss', 'o'),
                       (flat_df, '#2d6fa8', 'flat stake', 's')):
    d = df[df.final > 0]
    ax.plot(-d.max_dd_pct, d.final, marker=mk, ms=5, lw=1.7, color=c, label=lab)
ax.set_yscale('log'); ax.set_xlabel('worst fall (%), further left is better')
ax.set_ylabel('capital at the end (log scale)'); ax.legend(); plt.show()

## 8. The mean is the wrong number

On a strictly fair coin the rule cannot create an edge. Watch what it does to
the shape of the outcome instead, and note which statistic moves in which
direction.

In [ ]:
rng = np.random.default_rng(7)
N, T = 20000, 500
flips = rng.choice([-1.0, 1.0], size=(N, T))

flat_f = np.array([np.prod(1 + 0.01*row) for row in flips])
dbl_f = np.array([run_bets(row, 0.01)['final'] for row in flips])

print(pd.DataFrame([
    {'rule': 'flat stake', 'mean': flat_f.mean(), 'median': np.median(flat_f),
     'ruined_pct': (flat_f <= 0).mean()*100},
    {'rule': 'doubling', 'mean': dbl_f.mean(), 'median': np.median(dbl_f),
     'ruined_pct': (dbl_f <= 0).mean()*100},
]).to_string(index=False, float_format=lambda v: f'{v:10.4f}'))

print("\nThe doubling rule has the HIGHER mean. Losses stop at zero, and what")
print("lies beyond is not paid by the player, which lifts the average while the")
print("median sits at zero.")

## 9. Five things worth changing

1. **Lower `BASE_RISK` to 0.005.** On SPY that is the difference between zero
   and roughly 12 times the stake. Find where your own market's boundary sits.
2. **Set `SYMBOL = 'BTC-USD'`.** Longer losing runs, so the capacity
   `log2(B+1)` is exhausted sooner. Check the longest run before anything else.
3. **Cap the bet** at some multiple of remaining capital, which is what a
   broker does. The account then stops at a floor instead of zero, and the rule
   simply stops working rather than failing outright.
4. **Charge a cost per bet.** Both rules trade the same number of times, so
   costs hit them equally, but only one of them has a tail that reaches zero.
5. **Raise `WINDOW_BETS` to 600.** The share of windows ending at zero should
   rise toward the full-sample result. That gap is the whole argument of
   section 5.

Found something I got wrong? There is a form at the bottom of the article.